# Tutorial 13: A2A vs MCP - Architectural Comparison (Offline-First)

Capstone tutorial for `A2A/A2A_vs_MCP`, comparing tool-centric orchestration and agent-centric orchestration using the folder’s notebooks.

## 1) Where We Are in the Journey

`12_JWT_authentication_authorizatoin` secured orchestration endpoints.
This capstone explains execution architecture choices: pure MCP-style tool orchestration vs A2A coordination plus MCP execution.
It exists to clarify design boundaries and trade-offs.

## 2) What You Will Learn

- Differences between `without_a2a.ipynb` and `orchestrator2.ipynb` patterns
- Why one-tool execution fails for multi-step tasks
- How A2A planning enables chained workflows
- How MCP remains the execution substrate in both models

## 3) Why This Matters

Teams often confuse A2A and MCP as competing alternatives.
In practice they are complementary: A2A decides collaboration, MCP executes tools.
Choosing the wrong model leads to brittle orchestration under compound queries.

## 4) Core Concept Deep Dive

From source notebooks:
- `without_a2a.ipynb` discovers tools and executes one selected tool (single-hop limitation).
- `orchestrator2.ipynb` introduces planning with multiple sequential steps and result injection.
Trade-off: pure MCP orchestration is simpler for single-step requests; A2A+MCP handles compositional tasks robustly.

## 5) Code Walkthrough (Only `A2A/A2A_vs_MCP`)

- `without_a2a.ipynb`: discover tools -> find one tool -> execute once.
- `orchestratoin.ipynb`: basic route + invoke framing.
- `orchestrator2.ipynb`: explicit plan + sequential execute with intermediate-state injection.
This tutorial reproduces both approaches offline for side-by-side behavior comparison.

In [ ]:
import re

TOOLS = [
    {'name': 'calculate_interest', 'server': 'finance'},
    {'name': 'add', 'server': 'math'}
]

def invoke_tool(tool_name, args):
    if tool_name == 'calculate_interest':
        return {'status': 'success', 'result': (args['principal'] * args['rate'] * args['time']) / 100}
    if tool_name == 'add':
        return {'status': 'success', 'result': args['a'] + args['b']}
    return {'status': 'error'}

In [ ]:
def execute_without_a2a(query):
    numbers = list(map(float, re.findall(r'\d+\.?\d*', query)))
    tool = 'calculate_interest' if 'interest' in query else 'add' if 'add' in query else None
    if tool == 'calculate_interest':
        return invoke_tool('calculate_interest', {'principal': numbers[0], 'rate': numbers[1], 'time': numbers[2]})
    if tool == 'add':
        return invoke_tool('add', {'a': numbers[0], 'b': numbers[1]})
    return {'status': 'error', 'message': 'No tool'}

def plan_with_a2a(query):
    numbers = list(map(float, re.findall(r'\d+\.?\d*', query)))
    steps = []
    if 'interest' in query:
        steps.append({'tool': 'calculate_interest', 'args': {'principal': numbers[0], 'rate': numbers[1], 'time': numbers[2]}})
    if 'add' in query:
        steps.append({'tool': 'add', 'args': {'b': numbers[-1]}})
    return steps

def execute_with_a2a(query):
    steps = plan_with_a2a(query)
    result = None
    for step in steps:
        args = dict(step['args'])
        if step['tool'] == 'add':
            args['a'] = result
        res = invoke_tool(step['tool'], args)
        result = res['result']
    return {'status': 'success', 'result': result}

In [ ]:
q_single = 'calculate interest for 1000 at 5 for 2 years'
q_multi = 'calculate interest for 2000 at 10 for 1 year and then add 500'

print('WITHOUT A2A single:', execute_without_a2a(q_single))
print('WITHOUT A2A multi :', execute_without_a2a(q_multi))
print('WITH A2A single   :', execute_with_a2a(q_single))
print('WITH A2A multi    :', execute_with_a2a(q_multi))

## 6) System Flow Explanation

1. MCP-only path chooses one tool and executes once.
2. A2A+MCP path plans multiple steps.
3. Each step is executed via MCP-style tool invocation.
4. Intermediate result is injected into later steps.
5. Final output reflects composed multi-step reasoning.

## 7) Important Concepts You Might Miss

- MCP is execution protocol, not full orchestration policy.
- A2A supplies coordination logic across agents/tasks.
- Multi-step tasks reveal the limits of one-shot tool selection.

## 8) Common Mistakes / Pitfalls

- Treating A2A and MCP as mutually exclusive.
- Expecting single-tool execution to solve chained tasks.
- Omitting state passing in multi-step orchestration.

## 9) Key Takeaways

- A2A and MCP are complementary layers.
- MCP handles tool calls; A2A handles coordination and planning.
- Compound tasks generally require A2A-style workflow control.

## 10) What’s Next (Strict Preview)

Next step is production hardening across this full stack: retries, observability, auth policy, schema governance, and deployment reliability.
This matters because architecture concepts only deliver value when operationally robust in real environments.